In [7]:
# API documentation: https://wetterdienst.readthedocs.io/en/latest/usage/quickstart.html
from wetterdienst.provider.dwd.observation import DwdObservationRequest
import datetime as dt
from wetterdienst.provider.dwd.observation import DwdObservationMetadata
import pandas as pd
import polars as pl

# Look at all the data provided by dwd
metadata = DwdObservationMetadata.model_dump_json(indent=2)
#print(metadata)
# Find stations close to the Kirchentellinsfurt Wehr with relevant info
kirchentellinsfurt_wehr = (48.533, 9.183)

request_daily = DwdObservationRequest(
    parameters=[("daily", "climate_summary", "temperature_air_mean_2m"), ("daily", "climate_summary", "precipitation_amount"),
                ("daily", "climate_summary", "pressure_vapor_mean"), ("daily", "climate_summary", "humidity_relative_mean")],
    start=dt.datetime(2000, 1, 1),
    end=dt.datetime(2026, 8, 1)
)
# Get the stations within 50km of the Kirchentellinsfurt Wehr
stations = request_daily.filter_by_distance(latlon=kirchentellinsfurt_wehr, distance=50, unit="km")
stations_present_records = stations.df.filter(stations.df["end_timestamp"] > dt.datetime(2026, 8, 1, tzinfo=dt.timezone.utc))
pd.DataFrame(stations_present_records).to_csv("../data/intermediate/stations_climate_summary_50km_2000_2026.csv", index=False)
# Get the stations within 30km of the Kirchentellinsfurt Wehr
stations = request_daily.filter_by_distance(latlon=kirchentellinsfurt_wehr, distance=30, unit="km")
stations_present_records_30 = stations.df.filter(stations.df["end_timestamp"] > dt.datetime(2026, 8, 1, tzinfo=dt.timezone.utc))
pd.DataFrame(stations_present_records_30).to_csv("../data/intermediate/stations_climate_summary_30km_2000_2026.csv", index=False)


In [ ]:

# Get the data download for the stations that are present in the 30km radius.
relevant_requests = request_daily.filter_by_station_id(station_id=["03278", "04931", "02074", "06275", "03402"])
data = relevant_requests.values().all().df.drop_nulls("value")
# Convert data to long format (parameters as columns, timestamps and station_id as rows) for easier analysis
data_long = data.pivot(
    index=["station_id", "timestamp"],
    on="parameter",
    values="value"
)
pd.DataFrame(data_long).to_csv("../data/intermediate/weather_data_daily_2000_2026.csv", index=False)

In [9]:
# Count the number of observations per station_id and parameter
observation_counts = data.group_by(["station_id", "parameter"]).agg([pl.count("value").alias("observation_count")])
observation_counts # Note: no station has data for every day )


station_id,parameter,observation_count
str,str,u32
"""06275""","""temperature_air_mean_2m""",8067
"""03278""","""pressure_vapor_mean""",8305
"""02074""","""humidity_relative_mean""",9670
"""04931""","""precipitation_amount""",9706
"""02074""","""pressure_vapor_mean""",9670
…,…,…
"""06275""","""precipitation_amount""",8064
"""04931""","""humidity_relative_mean""",9623
"""02074""","""temperature_air_mean_2m""",9683


In [6]:

# Collect hourly data
request_hourly = DwdObservationRequest(
    parameters=[("hourly", "moisture", "temperature_air_2m"), ("hourly", "moisture", "pressure_vapor"), ("hourly", "moisture", "humidity_relative")],
    start=dt.datetime(2000, 1, 1),
    end=dt.datetime(2026, 8, 1)
)

relevant_requests_hourly = request_hourly.filter_by_station_id(station_id=["03278", "04931", "02074", "06275", "03402"])

data_hourly = relevant_requests_hourly.values().all().df.drop_nulls("value")
data_hourly_long = data_hourly.pivot(
    index=["station_id", "timestamp"],
    on="parameter",
    values="value"
)
pd.DataFrame(data_hourly_long).to_csv("../data/intermediate/weather_data_hourly_2000_2026.csv", index=False)

In [13]:
# Count the number of observations per station_id and parameter
observation_counts_hourly = data_hourly.group_by(["station_id", "parameter"]).agg([pl.count("value").alias("observation_count")])
observation_counts_hourly # data not recorded every hour in time window


station_id,parameter,observation_count
str,str,u32
"""03402""","""temperature_air_2m""",206228
"""02074""","""temperature_air_2m""",193019
"""02074""","""humidity_relative""",193019
"""03278""","""pressure_vapor""",198894
"""03278""","""temperature_air_2m""",198894
…,…,…
"""06275""","""pressure_vapor""",193096
"""06275""","""humidity_relative""",193096
"""03402""","""pressure_vapor""",206228
